# V11 Post-Processing - Complete Implementation

**Goal**: Boost V11 model LB score from 0.550 → 0.56-0.57

**Strategy**: 4 topology-safe operations
1. Threshold sweep (80% of gains)
2. Per-volume normalization
3. Connected component pruning
4. Ignore mask application

**Expected improvement**: +0.010 to +0.020 on LB

---

## 📦 Install Dependencies

In [ ]:
!pip install imagecodecs -q

## 🔧 Imports

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from scipy import ndimage
from typing import Optional, Tuple, Dict, List
from pathlib import Path
from tqdm.auto import tqdm
import json
import matplotlib.pyplot as plt

# Assuming your V11 training notebook has these
# from vesuvius_v11_topo_fixed import (
#     TopoPreservingUNet3D,
#     load_volume,
#     Config,
#     FOLD_SPLITS
# )

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

## 🏗️ V11PostProcessor Class

Core post-processing module with 4 topology-safe operations.

In [ ]:
class V11PostProcessor:
    """
    Topology-safe post-processing for V11 ink detection model.
    
    Pipeline:
    1. Optional per-volume normalization
    2. Global threshold
    3. Ignore mask application
    4. Connected component pruning (conservative)
    """
    
    def __init__(
        self,
        threshold: float = 0.50,
        normalize_per_volume: bool = True,
        min_component_size: int = 50,
        remove_border_tiny: bool = True,
        border_size_threshold: int = 20
    ):
        self.threshold = threshold
        self.normalize_per_volume = normalize_per_volume
        self.min_component_size = min_component_size
        self.remove_border_tiny = remove_border_tiny
        self.border_size_threshold = border_size_threshold
    
    def normalize_logits(
        self, 
        logits: np.ndarray, 
        valid_mask: Optional[np.ndarray] = None
    ) -> np.ndarray:
        """Normalize logits per volume to stabilize threshold."""
        if valid_mask is not None:
            valid_logits = logits[valid_mask > 0]
        else:
            valid_logits = logits
        
        if len(valid_logits) == 0:
            return logits
        
        mean = valid_logits.mean()
        std = valid_logits.std() + 1e-8
        
        return (logits - mean) / std
    
    def apply_threshold(
        self, 
        logits: np.ndarray, 
        threshold: float
    ) -> np.ndarray:
        """Apply sigmoid + threshold to get binary mask."""
        probs = 1.0 / (1.0 + np.exp(-logits))
        return (probs >= threshold).astype(np.uint8)
    
    def remove_small_components(
        self, 
        binary_mask: np.ndarray, 
        min_size: int,
        remove_border_tiny: bool = False,
        border_threshold: int = 20
    ) -> np.ndarray:
        """Remove small 3D connected components."""
        if min_size <= 0 and not remove_border_tiny:
            return binary_mask
        
        labeled, num_features = ndimage.label(binary_mask)
        
        if num_features == 0:
            return binary_mask
        
        component_sizes = np.bincount(labeled.ravel())
        
        # Identify border components
        border_components = set()
        if remove_border_tiny:
            D, H, W = binary_mask.shape
            border_components.update(np.unique(labeled[0, :, :]))
            border_components.update(np.unique(labeled[-1, :, :]))
            border_components.update(np.unique(labeled[:, 0, :]))
            border_components.update(np.unique(labeled[:, -1, :]))
            border_components.update(np.unique(labeled[:, :, 0]))
            border_components.update(np.unique(labeled[:, :, -1]))
            border_components.discard(0)
        
        # Determine which components to keep
        keep_mask = np.zeros(len(component_sizes), dtype=bool)
        keep_mask[0] = True  # background
        
        for comp_id in range(1, len(component_sizes)):
            size = component_sizes[comp_id]
            
            if size >= min_size:
                keep_mask[comp_id] = True
            elif remove_border_tiny and comp_id in border_components and size <= border_threshold:
                keep_mask[comp_id] = False
            else:
                keep_mask[comp_id] = True
        
        return keep_mask[labeled].astype(np.uint8)
    
    def process_volume(
        self,
        logits: np.ndarray,
        ignore_mask: Optional[np.ndarray] = None,
        return_probs: bool = False
    ) -> Dict[str, np.ndarray]:
        """Complete post-processing pipeline."""
        if logits.ndim == 4:
            logits = logits[0]
        if ignore_mask is not None and ignore_mask.ndim == 4:
            ignore_mask = ignore_mask[0]
        
        stats = {}
        
        # Build valid mask
        if ignore_mask is not None:
            valid_mask = (ignore_mask < 0.5).astype(np.uint8)
        else:
            valid_mask = np.ones_like(logits, dtype=np.uint8)
        
        stats['valid_voxels'] = valid_mask.sum()
        
        # Normalize
        if self.normalize_per_volume:
            logits = self.normalize_logits(logits, valid_mask)
            stats['normalized'] = True
        else:
            stats['normalized'] = False
        
        # Threshold
        binary = self.apply_threshold(logits, self.threshold)
        stats['voxels_before_mask'] = binary.sum()
        
        # Apply ignore mask
        binary = binary * valid_mask
        stats['voxels_after_mask'] = binary.sum()
        
        # Remove small components
        if self.min_component_size > 0 or self.remove_border_tiny:
            binary = self.remove_small_components(
                binary,
                min_size=self.min_component_size,
                remove_border_tiny=self.remove_border_tiny,
                border_threshold=self.border_size_threshold
            )
            stats['voxels_after_cc_pruning'] = binary.sum()
        
        result = {'binary': binary, 'stats': stats}
        
        if return_probs:
            probs = 1.0 / (1.0 + np.exp(-logits))
            result['probs'] = probs
        
        return result
    
    def sweep_threshold_on_val(
        self,
        val_logits_list: list,
        val_labels_list: list,
        val_ignore_list: list,
        thresholds: Optional[np.ndarray] = None
    ) -> Tuple[float, np.ndarray]:
        """Sweep threshold values on validation set."""
        if thresholds is None:
            thresholds = np.arange(0.30, 0.71, 0.02)
        
        dice_scores = []
        original_threshold = self.threshold
        
        for thresh in thresholds:
            self.threshold = thresh
            fold_dices = []
            
            for logits, labels, ignore in zip(val_logits_list, val_labels_list, val_ignore_list):
                result = self.process_volume(logits, ignore)
                binary = result['binary']
                
                pred = binary.astype(np.float32)
                gt = labels.astype(np.float32)
                
                if ignore is not None:
                    valid = (ignore < 0.5).astype(np.float32)
                    pred = pred * valid
                    gt = gt * valid
                
                inter = (pred * gt).sum()
                union = pred.sum() + gt.sum()
                dice = (2 * inter + 1e-5) / (union + 1e-5)
                fold_dices.append(dice)
            
            dice_scores.append(np.mean(fold_dices))
        
        best_idx = np.argmax(dice_scores)
        self.threshold = thresholds[best_idx]
        
        return thresholds[best_idx], np.array(dice_scores)

print('✅ V11PostProcessor class ready')

## 🔮 Full Volume Prediction

Sliding window inference with overlap averaging.

In [ ]:
@torch.no_grad()
def predict_full_volume(
    model,
    volume: np.ndarray,
    patch_size: Tuple[int, int, int] = (192, 192, 192),
    overlap: int = 32,
    batch_size: int = 4,
    device: str = 'cuda',
    use_bf16: bool = True
) -> np.ndarray:
    """
    Predict full volume with sliding window + overlap averaging.
    
    Returns:
        Logits (D, H, W) float32
    """
    model.eval()
    D, H, W = volume.shape
    pd, ph, pw = patch_size
    stride_d, stride_h, stride_w = pd - overlap, ph - overlap, pw - overlap
    
    # Pad if needed
    pad_d = max(0, pd - D)
    pad_h = max(0, ph - H)
    pad_w = max(0, pw - W)
    
    if pad_d > 0 or pad_h > 0 or pad_w > 0:
        volume = np.pad(
            volume,
            ((0, pad_d), (0, pad_h), (0, pad_w)),
            mode='reflect'
        )
        D, H, W = volume.shape
    
    logits_sum = np.zeros((D, H, W), dtype=np.float32)
    counts = np.zeros((D, H, W), dtype=np.float32)
    
    # Generate patches
    patches = []
    coords = []
    
    for z in range(0, D - pd + 1, stride_d):
        for y in range(0, H - ph + 1, stride_h):
            for x in range(0, W - pw + 1, stride_w):
                z_end = min(z + pd, D)
                y_end = min(y + ph, H)
                x_end = min(x + pw, W)
                
                patch = volume[z:z_end, y:y_end, x:x_end]
                
                if patch.shape != patch_size:
                    pad_z = pd - patch.shape[0]
                    pad_y = ph - patch.shape[1]
                    pad_x = pw - patch.shape[2]
                    patch = np.pad(
                        patch,
                        ((0, pad_z), (0, pad_y), (0, pad_x)),
                        mode='reflect'
                    )
                
                patches.append(patch)
                coords.append((z, y, x, z_end, y_end, x_end))
    
    # Process in batches
    dtype = torch.bfloat16 if use_bf16 else torch.float32
    
    for i in range(0, len(patches), batch_size):
        batch_patches = patches[i:i+batch_size]
        batch_coords = coords[i:i+batch_size]
        
        batch_tensor = torch.from_numpy(
            np.stack(batch_patches)[:, None, :, :, :]
        ).to(device, dtype=dtype)
        
        with torch.amp.autocast('cuda', enabled=use_bf16, dtype=dtype):
            out = model(batch_tensor)
            if isinstance(out, dict):
                out = out['output']
        
        logits_batch = out.float().cpu().numpy()[:, 0, :, :, :]
        
        for j, (z, y, x, z_end, y_end, x_end) in enumerate(batch_coords):
            actual_d = z_end - z
            actual_h = y_end - y
            actual_w = x_end - x
            
            logits_sum[z:z_end, y:y_end, x:x_end] += logits_batch[j, :actual_d, :actual_h, :actual_w]
            counts[z:z_end, y:y_end, x:x_end] += 1
    
    logits = logits_sum / np.maximum(counts, 1)
    
    if pad_d > 0 or pad_h > 0 or pad_w > 0:
        logits = logits[:D-pad_d, :H-pad_h, :W-pad_w]
    
    return logits

print('✅ predict_full_volume ready')

## 🎯 STEP 1: Load Trained Model

Load your best V11 checkpoint.

In [ ]:
def load_trained_model(checkpoint_path, cfg):
    """Load trained V11 model from checkpoint."""
    # Import from your training notebook
    from vesuvius_v11_topo_fixed import TopoPreservingUNet3D
    
    model = TopoPreservingUNet3D(
        in_ch=1,
        out_ch=1,
        features=cfg.FEATURES,
        nblocks=cfg.NBLOCKS,
        use_attention=cfg.USEATTENTION,
        use_hybridconv=cfg.USEHYBRIDCONV,
        use_surface_refinement=cfg.USESURFACEREFINEMENT,
        use_deep_supervision=False  # Turn off for inference
    ).to(device)
    
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    
    state_dict = checkpoint.get('model_state_dict', checkpoint)
    state_dict = {k.replace('_orig_mod.', ''): v for k, v in state_dict.items()}
    
    model.load_state_dict(state_dict, strict=False)
    model.eval()
    
    print(f"✅ Loaded model from {checkpoint_path}")
    print(f"   Checkpoint epoch: {checkpoint.get('epoch', 'unknown')}")
    print(f"   Best Dice: {checkpoint.get('best_dice', 0):.4f}")
    
    return model

# Example usage (uncomment and customize):
# from vesuvius_v11_topo_fixed import Config
# cfg = Config()
# model = load_trained_model(
#     checkpoint_path=Path('/kaggle/working/checkpoints/v11/fold0/best_model.pth'),
#     cfg=cfg
# )

## 📊 STEP 2: Predict Validation Set

Generate logits for all validation volumes (save for threshold sweep).

In [ ]:
# Uncomment and customize with your data
"""
from vesuvius_v11_topo_fixed import load_volume, FOLD_SPLITS

# Get validation IDs
val_ids = FOLD_SPLITS[0][1]  # Fold 0 validation

# Load volumes
print(f'Loading {len(val_ids)} validation volumes...')
val_volumes = []
val_labels = []
val_ignore = []

for vid in tqdm(val_ids):
    # Load image
    img = load_volume(Path('/kaggle/input/.../train/images') / f'{vid}.tif').astype(np.float32)
    img = (img - img.mean()) / (img.std() + 1e-8)
    val_volumes.append(img)
    
    # Load label
    lbl = load_volume(Path('/kaggle/input/.../train/labels') / f'{vid}.tif').astype(np.uint8)
    val_labels.append((lbl == 1).astype(np.uint8))
    val_ignore.append((lbl == 2).astype(np.uint8))

# Predict logits
print('Predicting validation logits...')
val_logits = []
for vol in tqdm(val_volumes, desc='Inference'):
    logits = predict_full_volume(
        model, vol,
        patch_size=(192, 192, 192),
        overlap=32,
        batch_size=4,
        device=device,
        use_bf16=True
    )
    val_logits.append(logits)

print(f'✅ Generated {len(val_logits)} validation predictions')
"""

## 🎯 STEP 3: Threshold Sweep (CRITICAL!)

**This is the most important step** - finds optimal threshold on validation.

Expected gain: **+0.010 on LB** (80% of total improvement)

In [ ]:
# Uncomment when you have val_logits, val_labels, val_ignore
"""
print('='*70)
print('THRESHOLD SWEEP')
print('='*70)

# Create post-processor
postprocessor = V11PostProcessor(
    threshold=0.50,  # Will be overwritten
    normalize_per_volume=True,
    min_component_size=50,
    remove_border_tiny=True,
    border_size_threshold=20
)

# Sweep thresholds
best_threshold, dice_scores = postprocessor.sweep_threshold_on_val(
    val_logits,
    val_labels,
    val_ignore,
    thresholds=np.arange(0.30, 0.71, 0.02)
)

print(f'\n✅ Best threshold: {best_threshold:.3f}')
print(f'   Validation Dice: {dice_scores.max():.4f}')

# Show top 5
thresholds = np.arange(0.30, 0.71, 0.02)
top5_idx = np.argsort(dice_scores)[-5:][::-1]

print('\nTop 5 thresholds:')
for idx in top5_idx:
    print(f'   {thresholds[idx]:.3f} → Dice {dice_scores[idx]:.4f}')

# Plot curve
plt.figure(figsize=(10, 5))
plt.plot(thresholds, dice_scores, 'o-', linewidth=2, markersize=6)
plt.axvline(best_threshold, color='red', linestyle='--', linewidth=2, label=f'Optimal: {best_threshold:.2f}')
plt.xlabel('Threshold', fontsize=12)
plt.ylabel('Validation Dice', fontsize=12)
plt.title('Threshold Sweep Results', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print('='*70)
"""

## 🔧 STEP 4: Tune Component Size (Optional)

Test different `min_component_size` values to reduce noise.

Expected gain: **+0.002 to +0.005** if output is noisy

In [ ]:
# Uncomment after threshold sweep
"""
print('\nComponent Size Ablation')
print('='*70)

for min_size in [0, 20, 50, 100, 150, 200]:
    postprocessor.min_component_size = min_size
    dices = []
    
    for logits, labels, ignore in zip(val_logits, val_labels, val_ignore):
        binary = postprocessor.process_volume(logits, ignore)['binary']
        
        pred = binary.astype(np.float32)
        gt = labels.astype(np.float32)
        
        if ignore is not None:
            valid = (ignore < 0.5).astype(np.float32)
            pred, gt = pred * valid, gt * valid
        
        inter = (pred * gt).sum()
        union = pred.sum() + gt.sum()
        dice = (2 * inter + 1e-5) / (union + 1e-5)
        dices.append(dice)
    
    print(f'  min_size={min_size:3d} → Dice {np.mean(dices):.4f}')

# Set to best value from above
# postprocessor.min_component_size = 50
"""

## 🚀 STEP 5: Generate Test Predictions

Apply optimized post-processing to test volumes.

In [ ]:
# Uncomment and customize
"""
print('='*70)
print('GENERATING TEST PREDICTIONS')
print('='*70)

test_dir = Path('/kaggle/input/vesuvius-challenge/test/volumes')
output_dir = Path('/kaggle/working/predictions_v11')
output_dir.mkdir(exist_ok=True)

test_files = sorted(test_dir.glob('*.tif'))
print(f'Found {len(test_files)} test volumes')

all_stats = {}

for test_file in tqdm(test_files, desc='Test inference'):
    print(f'\nProcessing {test_file.name}...')
    
    # Load and normalize
    test_vol = load_volume(test_file).astype(np.float32)
    test_vol = (test_vol - test_vol.mean()) / (test_vol.std() + 1e-8)
    
    # Predict logits
    test_logits = predict_full_volume(
        model, test_vol,
        patch_size=(192, 192, 192),
        overlap=32,
        batch_size=4,
        device=device,
        use_bf16=True
    )
    
    # Post-process
    result = postprocessor.process_volume(
        test_logits,
        ignore_mask=None,  # Add if you have test ignore masks
        return_probs=False
    )
    
    binary_pred = result['binary']
    
    # Save
    output_path = output_dir / f'{test_file.stem}_pred.npy'
    np.save(output_path, binary_pred)
    
    all_stats[test_file.stem] = result['stats']
    print(f'  Saved: {output_path.name}')
    print(f'  Stats: {result[\"stats\"]}')

# Save statistics
with open(output_dir / 'inference_stats.json', 'w') as f:
    json.dump(all_stats, f, indent=2)

print('\n' + '='*70)
print('DONE!')
print('='*70)
print(f'Predictions saved to: {output_dir}')
print(f'Total volumes: {len(test_files)}')
print(f'\nOptimized settings:')
print(f'  - Threshold: {postprocessor.threshold:.3f}')
print(f'  - Min component size: {postprocessor.min_component_size}')
print(f'  - Normalize per volume: {postprocessor.normalize_per_volume}')
print('='*70)
"""

## 📝 Summary

### Expected Improvements

| Step | LB Gain | Cumulative |
|------|---------|------------|
| Baseline (V11 @ epoch 45) | 0.550 | 0.550 |
| + Threshold sweep | +0.010 | **0.560** |
| + Per-volume norm | +0.005 | **0.565** |
| + CC pruning | +0.003 | **0.568** |

### Key Principles

1. ✅ **Threshold sweep is king** - 80% of gains
2. ✅ **Always validate on held-out data** - never training
3. ✅ **Keep it simple** - 4 operations only
4. ✅ **Topology-safe** - no morphology, no hole filling
5. ✅ **Work WITH your model** - it already learned topology

### Next Steps

1. Run threshold sweep on validation (15-30 min)
2. Optionally tune component size (10 min)
3. Generate test predictions (1-2 hours)
4. Submit to LB and compare

Good luck! 🚀